# How to read a JSON Lines file with Python

Load JSONL into pandas with Rugo, get the column types right, filter while reading, handle malformed lines, and query JSONL with SQL using Opteryx.

This notebook accompanies [How to read a JSON Lines file with Python](https://docs.opteryx.app/docs/guides/read-a-json-lines-file-with-python) on docs.opteryx.app.

In [ ]:
%pip install -q rugo opteryx-core pandas pyarrow

## Get the data

Download `space_missions.parquet` (4,630 orbital launch attempts) and convert it to JSONL with the `rugo` command.

In [ ]:
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/mabel-dev/opteryx-core/main/testdata/flat/space_missions/space_missions.parquet", "space_missions.parquet")

!rugo convert space_missions.parquet space_missions.jsonl
!head -n 2 space_missions.jsonl

## Load a JSONL file into pandas

Look at the types: `Price` should be a number.

In [ ]:
import pyarrow
from rugo.jsonl import read_jsonl

with read_jsonl("space_missions.jsonl") as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

df.dtypes

## Getting the types right

Types are inferred from the first 5 records. The first launches have no price, so `Price` falls back to text. Look at more records, or declare the type.

In [ ]:
with read_jsonl("space_missions.jsonl", infer_sample_size=500) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

df.dtypes

In [ ]:
with read_jsonl("space_missions.jsonl", explicit_schema={"Price": "DOUBLE"}) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

df.dtypes

In [ ]:
# a declared column is parsed strictly
with open("na.jsonl", "w") as f:
    f.write('{"Price": 1.5}\n{"Price": "N/A"}\n')

try:
    with read_jsonl("na.jsonl", explicit_schema={"Price": "DOUBLE"}) as reader:
        list(reader)
except ValueError as e:
    print(e)

## Timestamps

Timestamps arrive as text. These carry a `+00:00` offset, so convert them after reading.

In [ ]:
import pandas as pd

df["Lauched_at"] = pd.to_datetime(df["Lauched_at"], utc=True)
df["Lauched_at"].head()

## Read only the columns and rows you need

Non-matching records never become columns.

In [ ]:
with read_jsonl(
    "space_missions.jsonl",
    columns=["Company", "Mission", "Price"],
    predicates=[("Company", "==", "SpaceX"), ("Price", "<", 60)],
    infer_sample_size=500,
) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

len(df)

In [ ]:
# a predicate value must match the column's JSON type; it is never converted
try:
    with read_jsonl("space_missions.jsonl", predicates=[("Price", "<", "60")]) as reader:
        list(reader)
except ValueError as e:
    print(e)

## Malformed lines

By default a bad line fails the read. `fail_on_error=False` skips it.

In [ ]:
with open("events.jsonl", "w") as f:
    f.write('{"a": 1}\n{"a": 2\n{"a": 3}\n')

try:
    with read_jsonl("events.jsonl") as reader:
        list(reader)
except ValueError as e:
    print(e)

In [ ]:
with read_jsonl("events.jsonl", fail_on_error=False) as reader:
    print(pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas())

## Query with SQL: Opteryx

`READ_JSONL` uses the same inference, with the same options.

In [ ]:
import opteryx

morsels = opteryx.session().execute_to_morsels("""
    SELECT Mission_Status, COUNT(*) AS launches
      FROM READ_JSONL('space_missions.jsonl')
     WHERE Company = 'SpaceX'
     GROUP BY Mission_Status
     ORDER BY launches DESC
""")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()

In [ ]:
morsels = opteryx.session().execute_to_morsels("""
    SELECT COUNT(*) AS cheap_launches
      FROM READ_JSONL('space_missions.jsonl', infer_sample_size => 500)
     WHERE Price < 60
""")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()

In [ ]:
morsels = opteryx.session().execute_to_morsels("""
    SELECT EXTRACT(year FROM CAST(Lauched_at AS TIMESTAMP)) AS year, COUNT(*) AS launches
      FROM READ_JSONL('space_missions.jsonl')
     GROUP BY year
     ORDER BY launches DESC
     LIMIT 5
""")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()

## Convert to Parquet

Fix the types once, with Rugo reading the JSONL and writing the Parquet.

In [ ]:
from rugo import parquet

with read_jsonl("space_missions.jsonl", explicit_schema={"Price": "DOUBLE"}) as reader:
    with open("space_missions_typed.parquet", "wb") as f:
        with parquet.open_parquet_writer(f.write) as writer:
            for morsel in reader:
                writer.write_row_group(morsel)

In [ ]:
!rugo schema space_missions_typed.parquet